In [ ]:
from pathlib import Path
import yaml
import json
import re
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd

# load the site config
with open("site_config.yaml") as f:
    config = yaml.safe_load(f)

# project directory
project_directory = Path(config['project_directory'])

In [ ]:
def strip_html(cell: str) -> str:
    # collapse <sub>x</sub> into a plain suffix (e.g. "R<sub>merge</sub>" -> "R_merge")
    cell = re.sub(r"<sub>(.*?)</sub>", r"_\1", cell)
    # drop any remaining tags (<strong>, </strong>, etc.)
    return re.sub(r"<[^>]+>", "", cell).strip()

def extract_summary_table(scaling_report):
    summary_table = scaling_report['scaling_tables']['overall']
    rows = [[strip_html(cell) for cell in r] for r in summary_table]
    header, *body = rows
    df = pd.DataFrame(body, columns=header).set_index(header[0])
    return df

def extract_frame_range(scaling_report):
    return scaling_report['image_range_tables'][0][1][1]

rows = []
for p in project_directory.glob("*/*/dials/truncated/scaled_truncated.json"):
    synchrotron, date_str = p.parts[-5].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-4]}
    with open(p,'r') as f:
        scaling_report = json.load(f)
    summary_table = extract_summary_table(scaling_report)
    if row['crystal'].endswith('merge'):
        row['crystal'] = 'merged'
        row['Frames'] = ''
    else:
        frame_range = extract_frame_range(scaling_report)
        #degrees = (frame_range[1]-frame_range[0])/10
        #row['frame range'] = f'{frame_range[0]}-{frame_range[1]} ({round(degrees)}˚)'
        row['Frames'] = f'{frame_range[0]}-{frame_range[1]}'
    resol = summary_table['High resolution']['Resolution (Å)']
    row['CC_½'] = summary_table['High resolution']['CC_½']
    row['Completeness'] = summary_table['High resolution']['Completeness']
    rows.append(row)
    
df_scaling = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_scaling

In [ ]:
rows = []
for p in project_directory.glob("*/*/dials/estimate_resolution/dials.estimate_resolution.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-5].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-4]}
    m = re.search(r'Resolution cc_half:\s*([0-9.]+)', t, re.M)
    row['Resolution'] = m.group(1) if m else None
    rows.append(row)
df_resol = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_resol

In [ ]:
# integration stats (mosaicity, cell params)
pats = {
    "Mosaicity": r"sigma m:\s*([0-9.]+)",
    "Unit Cell": r"Unit cell:\s*(.+)",
}
rows = []
for p in project_directory.glob("*/*/dials/dials.integrate.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-4].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-3]}
    for k, pat in pats.items():
        m = re.search(pat, t, re.M)
        row[k] = m.group(1) if m else None
    rows.append(row)
    # fix the unit cell cloumn
    if row["Unit Cell"] is not None:
        row["Unit Cell"] = tuple(
            float(re.sub(r"\(.*?\)", "", x).strip())
            for x in row["Unit Cell"].split(",")
        )
df_integrate = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_integrate['a'] = df_integrate['Unit Cell'].apply(lambda t: round(t[0],2))
df_integrate['c'] = df_integrate['Unit Cell'].apply(lambda t: round(t[2],2))
df_integrate['Mosaicity'] = pd.to_numeric(df_integrate['Mosaicity']).round(3)
#df_integrate = df_integrate.rename(columns={'Mosaicity': 'Mosaicity (˚)'})
df_integrate = df_integrate.drop(columns=['Unit Cell'])
df_integrate

In [ ]:
df = df_scaling.join(df_resol).join(df_integrate)
df = df[['Frames','Mosaicity','Resolution','a','c','CC_½','Completeness']]
df.columns = pd.MultiIndex.from_tuples([
     ('Frames', ''),                 # Flat column (has a blank placeholder level)
     ('Mosaicity', ''),
     ('Resolution', ''),
     ('Unit Cell', 'a'),
     ('Unit Cell', 'b'),
     ('Outer shell (1.27-1.25 Å)', 'CC_½'),       # Nested column
     ('Outer shell (1.27-1.25 Å)', 'Completeness')        # Nested column
])
#df.style.format(na_rep="")
df

In [ ]:
# export
df.to_csv("figures/source_data/dials_stats.csv")

## Humidity tests

In [ ]:
# merge stats (Wilson B)
pats = {
    "Wilson B": r"Wilson B factor\s*([0-9.]+)",
}
rows = []
for p in project_directory.glob("*/*/dials/dials.merge.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-4].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-3]}
    for k, pat in pats.items():
        m = re.search(pat, t, re.M)
        row[k] = m.group(1) if m else None
    rows.append(row)

df_merge = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_merge

In [ ]:
df = df_integrate.join(df_merge,how='outer')
df = df.loc['ALS'] # drop the CHESS data 
df = df.drop([('2025-10-15','mac1_7')], axis=0) # drop the mac1_7 dataset (radiation damage test)
df = df.drop('2026-04-23',axis=0) # drop the small crystal dataset
df

In [ ]:
df.to_csv("figures/source_data/dials_stats_humidity.csv")